# Load Pattern, Flexibility, and DER Opportunity Analysis Engine
## Stage 1 + Stage 2: Architecture/Schema/Ingestion/Daily Profiles + Analytical Engine

Implements Implementation Handoff Specification v0.5, Stage 1 and
Stage 2 scope (per Section 59's recommended staged delivery):

**Stage 1**: architecture, canonical schema, configuration parser,
ingestion, validation, aggregation, time handling, daily profiles.

**Stage 2**: features (time-of-day segments, temperature change-point
model), demand classification, ramps/peaks/valleys, peak events,
load-shape classification, daily-profile clustering (absolute +
normalized), pattern discovery, meter coincidence.

**Not in this notebook** (Stage 3): opportunity/scenario engine
(shedding/shifting/modulation/solar/storage), TOU, user-defined
searches, visualization, full export suite.

In [1]:
import os
from pathlib import Path

# ============================================================================
# CONFIGURATION — the only cell you need to change to analyze a different
# meter, portfolio, or dataset. Everything below is driven entirely by what
# this config file declares (meters, groups, portfolio membership, input
# file path, interval resolution, etc.) — no meter IDs or entity names are
# hardcoded anywhere else in this notebook.
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
parent_dir = os.path.abspath("..")
CONFIG_PATH = os.path.join(parent_dir, "config", "mcdonough_hall_configuration.toml")

In [2]:
import sys
import logging

import pandas as pd

INPUT_DATA_DIR = PROJECT_ROOT / "data" / "input"

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src import config as cfg_mod
from src import ingestion
from src import timeproc
from src import missing
from src import entities
from src import profiles

pd.set_option("display.width", 120)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Load and Validate Configuration (Section 35-36)

In [3]:
config = cfg_mod.load_configuration(CONFIG_PATH)

findings = cfg_mod.validate_configuration(config)
for f in findings:
    print(f"[{f.severity}] {f.section}: {f.message}")
cfg_mod.raise_if_errors(findings)
print("\nConfiguration valid. Proceeding.")


Configuration valid. Proceeding.


## 2. Load Input Data and Map to Canonical Schema (Section 3, 37)

In [4]:
raw_df = ingestion.load_input_data(config, base_dir=parent_dir)
canonical_df = ingestion.map_to_canonical_schema(raw_df, config)
canonical_df["timestamp"] = pd.to_datetime(canonical_df["timestamp"])

print(f"Raw rows loaded: {len(raw_df)}")
canonical_df.head()

/var/folders/cl/bn539g3s0_b18p8wwzsg6xmr0000gn/T/ipykernel_43491/429725942.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  canonical_df["timestamp"] = pd.to_datetime(canonical_df["timestamp"])


Raw rows loaded: 74835


,timestamp,meter_id,demand_kw,temperature_f
0,2024-07-29 11:15:00,KZD390965687-1,692.72,82.0
1,2024-07-29 11:30:00,KZD390965687-1,693.76,82.0
2,2024-07-29 11:45:00,KZD390965687-1,697.76,82.0
3,2024-07-29 12:00:00,KZD390965687-1,699.60,82.0
4,2024-07-29 12:15:00,KZD390965687-1,702.64,84.9


In [5]:
data_findings = ingestion.validate_input_data(canonical_df, config)
for f in data_findings:
    print(f"[{f.severity}] {f.section}: {f.message}")

# Duplicate (meter_id, timestamp) records are detected above as ERROR
# (Section 37), then deterministically resolved here (keep first
# occurrence) before any other ERROR-severity finding is treated as fatal.
# This keeps detection and remediation both visible and auditable.
n_before = len(canonical_df)
canonical_df = canonical_df.drop_duplicates(subset=["meter_id", "timestamp"], keep="first")
print(f"Dropped {n_before - len(canonical_df)} duplicate (meter_id, timestamp) row(s).")

remaining_findings = [f for f in data_findings if f.section != "ingestion" or "duplicate" not in f.message.lower()]
errors = [f for f in remaining_findings if f.severity == "ERROR"]
if config.get("validation", {}).get("strict") and any(
    f.severity == "WARNING" for f in remaining_findings
):
    errors += [f for f in remaining_findings if f.severity == "WARNING"]
if errors:
    raise RuntimeError("Data validation failed; see findings above")

[WARNING] ingestion: 5576 duplicate (meter_id, timestamp) record(s) detected
Dropped 5576 duplicate (meter_id, timestamp) row(s).


## 3. Detect Time Resolution (Section 6)

In [6]:
resolution_info = timeproc.detect_time_resolution(
    canonical_df["timestamp"], meter_id=canonical_df["meter_id"]
)
resolution_info

{'expected_interval_minutes': 15,
 'is_mixed_resolution': False,
 'n_irregular_gaps': 4,
 'n_duplicate_timestamps': 0,
 'per_meter': {'KZD390965687-1': 15}}

In [7]:
configured_resolution = config["data"]["time"]["resolution"]
if configured_resolution == "auto":
    interval_minutes = resolution_info["expected_interval_minutes"]
else:
    interval_minutes = int(configured_resolution.replace("min", ""))
print(f"Using native interval: {interval_minutes} minutes")

Using native interval: 15 minutes


## 4. Handle Missing Data (Section 7) — per meter

In [8]:
processed_df = missing.handle_missing_data(
    canonical_df, interval_minutes, config["data"]["missing"]
)
print(processed_df["data_quality_flag"].value_counts())

# Per-meter + portfolio-wide missing-data awareness (Section 7)
missing_data_summary = missing.summarize_missing_data(processed_df)
missing_intervals = missing.missing_intervals_detail(processed_df)
missing.log_missing_data_summary(missing_data_summary)

quality_output_dir = PROJECT_ROOT / config["output"]["output_directory"] / "quality"
quality_output_dir.mkdir(parents=True, exist_ok=True)
missing_data_summary.to_csv(quality_output_dir / "missing_data_summary.csv", index=False)
missing_intervals.to_csv(quality_output_dir / "missing_intervals_detail.csv", index=False)

missing_data_summary

data_quality_flag
observed        69259
missing           472
interpolated       16
Name: count, dtype: int64


INFO src.missing: data_quality meter=KZD390965687-1 intervals=69747 observed=69259 (99.30%) interpolated=16 (0.02%) missing=472 (0.68%) gap_events=4 max_gap=384 intervals [2026-07-16 00:00:00 .. 2026-07-19 23:45:00]


INFO src.missing: data_quality meter=PORTFOLIO intervals=69747 observed=69259 (99.30%) interpolated=16 (0.02%) missing=472 (0.68%) gap_events=4 max_gap=384 intervals [2026-07-16 00:00:00 .. 2026-07-19 23:45:00]


,meter_id,n_intervals,n_observed,pct_observed,n_interpolated,pct_interpolated,n_missing,pct_missing,n_gap_events,max_gap_intervals,max_gap_start,max_gap_end
0,KZD390965687-1,69747,69259,99.3,16,0.02,472,0.68,4,384,2026-07-16,2026-07-19 23:45:00
1,PORTFOLIO,69747,69259,99.3,16,0.02,472,0.68,4,384,2026-07-16,2026-07-19 23:45:00


## 5. Build Calendar Features (Section 9)

In [9]:
calendar_cfg = config.get("calendar", {})
season_map_cfg = config.get("calendar", {}).get("seasons", {})
season_by_month = {int(k): v for k, v in season_map_cfg.items()} if season_map_cfg else None

calendar_features = timeproc.build_calendar_features(
    processed_df["timestamp"],
    holidays=calendar_cfg.get("holidays"),
    season_by_month=season_by_month,
)
processed_df = pd.concat(
    [processed_df.reset_index(drop=True), calendar_features.reset_index(drop=True)], axis=1
)
processed_df[["timestamp", "meter_id", "day_type", "season"]].head()

,timestamp,meter_id,day_type,season
0,2024-07-29 11:15:00,KZD390965687-1,weekday,summer
1,2024-07-29 11:30:00,KZD390965687-1,weekday,summer
2,2024-07-29 11:45:00,KZD390965687-1,weekday,summer
3,2024-07-29 12:00:00,KZD390965687-1,weekday,summer
4,2024-07-29 12:15:00,KZD390965687-1,weekday,summer


## 6. Calculate Interval Energy (Section 2.2)

In [10]:
processed_df["energy_kwh"] = timeproc.calculate_interval_energy(
    processed_df["analysis_demand_kw"], interval_minutes
)
processed_df[["timestamp", "meter_id", "analysis_demand_kw", "energy_kwh"]].head()

,timestamp,meter_id,analysis_demand_kw,energy_kwh
0,2024-07-29 11:15:00,KZD390965687-1,692.72,173.18
1,2024-07-29 11:30:00,KZD390965687-1,693.76,173.44
2,2024-07-29 11:45:00,KZD390965687-1,697.76,174.44
3,2024-07-29 12:00:00,KZD390965687-1,699.60,174.90
4,2024-07-29 12:15:00,KZD390965687-1,702.64,175.66


## 7. Resolve Meter Groups and Portfolio (Section 4-5)

In [11]:
resolved_groups = entities.build_meter_groups(config)
for name, members in resolved_groups.items():
    print(f"{name:15s} -> {members}")

portfolio_meters = entities.build_portfolio_meters(config)
print(f"\nPortfolio -> {portfolio_meters}")


Portfolio -> ['KZD390965687-1']


## 8. Aggregate Entity Load (individual meters, groups, portfolio) — Section 5, sum not average

In [12]:
entity_load_tables = {}

for meter_id in [m["meter_id"] for m in config["meters"]]:
    single = processed_df[processed_df["meter_id"] == meter_id][
        ["timestamp", "analysis_demand_kw"]
    ].rename(columns={"analysis_demand_kw": "demand_kw"})
    single["n_meters_reporting"] = single["demand_kw"].notna().astype(int)
    entity_load_tables[meter_id] = single

for group_name, members in resolved_groups.items():
    entity_load_tables[group_name] = entities.aggregate_entity_load(processed_df, members)

entity_load_tables["Portfolio"] = entities.aggregate_entity_load(processed_df, portfolio_meters)

print("Entities constructed:", list(entity_load_tables.keys()))
entity_load_tables["Portfolio"].head()

Entities constructed: ['KZD390965687-1', 'Portfolio']


,timestamp,demand_kw,n_meters_reporting
0,2024-07-29 11:15:00,692.72,1
1,2024-07-29 11:30:00,693.76,1
2,2024-07-29 11:45:00,697.76,1
3,2024-07-29 12:00:00,699.60,1
4,2024-07-29 12:15:00,702.64,1


## 9. Construct Daily Profiles and Calculate Daily Features (Section 8, 10-12)

In [13]:
daily_profile_tables = {}
daily_feature_tables = {}

for entity_id, load_df in entity_load_tables.items():
    daily = profiles.construct_daily_profiles(load_df, interval_minutes, entity_id=entity_id)
    daily_normalized = profiles.normalize_daily_profiles(daily)
    feats = profiles.calculate_daily_features(daily, interval_minutes)
    daily_profile_tables[entity_id] = daily_normalized
    daily_feature_tables[entity_id] = feats

daily_feature_tables["Portfolio"]

,entity_id,date,mean_demand_kw,maximum_demand_kw,minimum_demand_kw,daily_energy_kwh,peak_time,load_factor,peak_to_average_ratio,standard_deviation_kw,coefficient_of_variation,is_complete_day
0,Portfolio,2024-07-29,735.807059,880.08,665.28,9381.54,14:00:00,0.836068,1.196074,71.487008,0.097155,False
1,Portfolio,2024-07-30,715.230833,896.64,668.48,17165.54,13:30:00,0.797679,1.253637,65.350917,0.091370,True
2,Portfolio,2024-07-31,732.398333,867.44,656.48,17577.56,14:45:00,0.844322,1.184383,35.266783,0.048152,True
3,Portfolio,2024-08-01,722.860000,856.16,658.88,17348.64,23:00:00,0.844305,1.184406,50.485457,0.069841,True
4,Portfolio,2024-08-02,749.603333,830.56,644.56,17990.48,09:15:00,0.902528,1.107999,34.488476,0.046009,True
...,...,...,...,...,...,...,...,...,...,...,...,...
722,Portfolio,2026-07-21,734.161667,809.76,670.64,17619.88,13:45:00,0.906641,1.102972,36.609451,0.049866,True
723,Portfolio,2026-07-22,694.440833,769.36,594.64,16666.58,11:45:00,0.902621,1.107884,36.225095,0.052164,True
724,Portfolio,2026-07-23,585.325000,627.92,532.48,14047.80,01:30:00,0.932165,1.072772,23.770246,0.040610,True
725,Portfolio,2026-07-24,572.772500,624.96,507.76,13746.54,11:00:00,0.916495,1.091114,26.804211,0.046797,True


## 10. Stage 1 Summary (Section 52, Stage-1 subset)

In [14]:
n_complete = int(daily_feature_tables["Portfolio"]["is_complete_day"].sum())
n_days_total = len(daily_feature_tables["Portfolio"])

print("=" * 72)
print("STAGE 1 ANALYTICAL SUMMARY")
print("=" * 72)
print(f"Input dataset:        {config['data']['input']['file_path']}")
print(f"Native resolution:    {interval_minutes} minutes")
print(f"Meters analyzed:      {[m['meter_id'] for m in config['meters']]}")
print(f"Groups analyzed:      {list(resolved_groups.keys())}")
print(f"Portfolio meters:     {portfolio_meters}")
print(f"Portfolio days:       {n_days_total} total, {n_complete} complete, "
      f"{n_days_total - n_complete} incomplete")
print(f"Missing-data policy:  interpolation_enabled="
      f"{config['data']['missing']['interpolation_enabled']}, "
      f"max_gap={config['data']['missing']['max_interpolation_intervals']} intervals")
print("Data quality (all meters, native resolution):")
print(processed_df["data_quality_flag"].value_counts().to_string())
print("=" * 72)
print("Stage 1 complete. Ready for Stage 2 (features/peaks/shapes/clustering/patterns).")

STAGE 1 ANALYTICAL SUMMARY
Input dataset:        data/input/Green Button Data McDonough Hall - Updated.csv
Native resolution:    15 minutes
Meters analyzed:      ['KZD390965687-1']
Groups analyzed:      []
Portfolio meters:     ['KZD390965687-1']
Portfolio days:       727 total, 721 complete, 6 incomplete
Missing-data policy:  interpolation_enabled=True, max_gap=4 intervals
Data quality (all meters, native resolution):
data_quality_flag
observed        69259
missing           472
interpolated       16
Stage 1 complete. Ready for Stage 2 (features/peaks/shapes/clustering/patterns).


## 11. Export Stage 1 Outputs (Section 51 — Stage-1 subset)

In [15]:
output_dir = PROJECT_ROOT / config["output"]["output_directory"]
(output_dir / "observations").mkdir(parents=True, exist_ok=True)
(output_dir / "daily").mkdir(parents=True, exist_ok=True)

processed_df.to_csv(output_dir / "observations" / "native_resolution_observations.csv", index=False)
for entity_id, feats in daily_feature_tables.items():
    feats.to_csv(output_dir / "daily" / f"daily_features_{entity_id}.csv", index=False)

print(f"Exported to: {output_dir}")

Exported to: /Users/egeriicw/Personal-Projects/load_flexibility_and_der_analysis/output


# Stage 2: Analytical Engine
Features, peaks/valleys/ramps, load-shape classification, clustering,
pattern discovery, and meter coincidence (Sections 12-13, 14-17, 18,
19-20, 21, 22).

In [16]:
from src import features as features_mod
from src import peaks as peaks_mod
from src import shapes as shapes_mod
from src import clustering as clustering_mod
from src import patterns as patterns_mod
from src import coincidence as coincidence_mod

(output_dir / "peaks").mkdir(parents=True, exist_ok=True)
(output_dir / "clusters").mkdir(parents=True, exist_ok=True)
(output_dir / "patterns").mkdir(parents=True, exist_ok=True)

## 12. Time-of-Day Segment and Temperature Features (Section 12-13)

In [17]:
segment_feature_tables = {}
for entity_id, daily in daily_profile_tables.items():
    segment_feature_tables[entity_id] = features_mod.calculate_segment_features(daily)

segment_feature_tables["Portfolio"].head()

,entity_id,date,morning_peak_kw,midday_peak_kw,afternoon_peak_kw,evening_peak_kw,overnight_mean_kw,nighttime_mean_kw,daytime_mean_kw
0,Portfolio,2024-07-29,NaN,709.84,880.08,872.00,731.9500,731.9500,736.524651
1,Portfolio,2024-07-30,718.40,896.64,895.92,886.56,688.3775,688.3775,728.657500
2,Portfolio,2024-07-31,733.60,825.92,867.44,761.84,725.0925,725.0925,736.051250
3,Portfolio,2024-08-01,811.92,786.24,783.68,756.88,688.0700,688.0700,740.255000
4,Portfolio,2024-08-02,830.56,782.72,772.48,760.48,760.4775,760.4775,744.166250


In [18]:
# Change-point (balance-point) cooling model per meter, weekday-only to
# avoid the weekday/weekend load-level swing confounding the fit
# (Section 13 — STATISTICAL method; correlation, not causation).
change_point_results = {}
for meter_id in [m["meter_id"] for m in config["meters"]]:
    m_obs = processed_df[(processed_df["meter_id"] == meter_id) & (processed_df["is_weekday"])]
    daily_temp = m_obs.groupby(m_obs["timestamp"].dt.date)["temperature_f"].mean()
    daily_demand = m_obs.groupby(m_obs["timestamp"].dt.date)["analysis_demand_kw"].mean()
    change_point_results[meter_id] = features_mod.fit_change_point_model(
        daily_temp.values, daily_demand.values
    )

pd.DataFrame(change_point_results).T

,baseload_kw,slope_kw_per_f,breakpoint_f,r_squared,n_points,method,success
KZD390965687-1,504.289261,12.349613,61.0,0.509192,519,change_point_regression,True


In [19]:
# 5-parameter heating+cooling change-point model (Section 13 extension
# point): generalizes the 3P cooling-only model above with an added
# heating side, useful when a meter shows a visible heating response
# (electric heat, or a fossil-fuel meter) in addition to cooling.
change_point_5p_results = {}
for meter_id in [m["meter_id"] for m in config["meters"]]:
    m_obs = processed_df[(processed_df["meter_id"] == meter_id) & (processed_df["is_weekday"])]
    daily_temp = m_obs.groupby(m_obs["timestamp"].dt.date)["temperature_f"].mean()
    daily_demand = m_obs.groupby(m_obs["timestamp"].dt.date)["analysis_demand_kw"].mean()
    change_point_5p_results[meter_id] = features_mod.fit_change_point_model_5p(
        daily_temp.values, daily_demand.values
    )

pd.DataFrame(change_point_5p_results).T

,base_kw,heating_slope_kw_per_f,heating_breakpoint_f,cooling_slope_kw_per_f,cooling_breakpoint_f,r_squared,n_points,method,success
KZD390965687-1,450.031352,1.483471,50.0,12.59175,57.0,0.554513,519,change_point_regression_5p,True


### Full change-point model family and automatic model selection

Rather than assuming a priori which change-point model form applies to
a given meter, fit the full ASHRAE GL14 / IPMVP family — 2P (no
breakpoint), 3P heating, 3P cooling, 4P (shared breakpoint), 5P
(independent heating+cooling breakpoints) — and let adjusted R² (which
penalizes the extra parameters of the more complex models) pick the
best-supported one per meter.

In [20]:
change_point_family_results = {}
for meter_id in [m["meter_id"] for m in config["meters"]]:
    m_obs = processed_df[(processed_df["meter_id"] == meter_id) & (processed_df["is_weekday"])]
    daily_temp = m_obs.groupby(m_obs["timestamp"].dt.date)["temperature_f"].mean()
    daily_demand = m_obs.groupby(m_obs["timestamp"].dt.date)["analysis_demand_kw"].mean()
    change_point_family_results[meter_id] = features_mod.select_best_change_point_model(
        daily_temp.values, daily_demand.values
    )

pd.DataFrame(
    {
        meter_id: {
            "selected_model": r["selected_model"],
            **{f"selected.{k}": v for k, v in (r["selected"] or {}).items()},
        }
        for meter_id, r in change_point_family_results.items()
    }
).T

,selected_model,selected.base_kw,selected.heating_slope_kw_per_f,selected.cooling_slope_kw_per_f,selected.breakpoint_f,selected.r_squared,selected.n_points,selected.method,selected.success
KZD390965687-1,4p,444.920135,1.386251,12.846068,57.0,0.55441,519,change_point_regression_4p,True


## 13. Demand Classification, Ramps, Peaks/Valleys, Peak Events (Section 14-17)

In [21]:
demand_thresholds = config["analysis"]["demand"]["thresholds_kw"]
top_percentiles = config["analysis"]["demand"]["top_percentiles"]
top_n_hours = config["analysis"]["demand"]["top_n_hours"]
allowable_gap = config["analysis"]["peak_events"]["allowable_gap_intervals"]

peak_events_by_entity = {}
ramp_tables = {}

for entity_id, load_df in entity_load_tables.items():
    load_df = load_df.sort_values("timestamp").reset_index(drop=True)
    ramps = peaks_mod.detect_ramps(load_df["demand_kw"])
    pv = peaks_mod.detect_local_peaks_valleys(load_df["demand_kw"])
    ramp_tables[entity_id] = pd.concat(
        [load_df[["timestamp", "demand_kw"]], ramps, pv], axis=1
    )

    obs_for_events = load_df.rename(columns={"demand_kw": "demand_kw"})
    meets = obs_for_events["demand_kw"] >= demand_thresholds[0]
    events = peaks_mod.build_peak_events(
        obs_for_events, meets, allowable_gap, entity_id, f"threshold_{demand_thresholds[0]}"
    )
    peak_events_by_entity[entity_id] = events

peak_events_by_entity["Portfolio"]

,event_id,entity_id,peak_definition,start_time,end_time,duration_hours,maximum_demand_kw,mean_demand_kw,minimum_demand_kw,n_intervals
0,Portfolio_threshold_500_0000,Portfolio,threshold_500,2024-07-29 11:15:00,2024-08-20 23:00:00,539.75,896.64,670.992741,522.88,2160
1,Portfolio_threshold_500_0001,Portfolio,threshold_500,2024-08-21 10:15:00,2024-08-22 01:00:00,14.75,765.92,622.921333,539.44,60
2,Portfolio_threshold_500_0002,Portfolio,threshold_500,2024-08-22 08:30:00,2024-09-04 00:30:00,304.00,1026.96,693.843287,532.40,1217
3,Portfolio_threshold_500_0003,Portfolio,threshold_500,2024-09-04 11:30:00,2024-09-05 03:45:00,16.25,808.48,670.915152,587.36,66
4,Portfolio_threshold_500_0004,Portfolio,threshold_500,2024-09-05 05:30:00,2024-09-10 05:45:00,120.25,782.24,636.677842,520.16,482
...,...,...,...,...,...,...,...,...,...,...
488,Portfolio_threshold_500_0488,Portfolio,threshold_500,2026-06-30 05:00:00,2026-06-30 23:00:00,18.00,793.28,639.215342,515.52,73
489,Portfolio_threshold_500_0489,Portfolio,threshold_500,2026-07-01 05:00:00,2026-07-01 23:15:00,18.25,909.84,714.267027,560.80,74
490,Portfolio_threshold_500_0490,Portfolio,threshold_500,2026-07-02 05:00:00,2026-07-07 00:45:00,115.75,908.96,710.385862,568.72,464
491,Portfolio_threshold_500_0491,Portfolio,threshold_500,2026-07-08 00:00:00,2026-07-16 00:45:00,192.75,933.92,689.249217,557.12,772


## 14. Load-Shape Classification (Section 18)

In [22]:
shape_tables = {}
for entity_id in daily_profile_tables:
    daily = daily_profile_tables[entity_id].copy()
    pv = peaks_mod.detect_local_peaks_valleys(daily["demand_kw"])
    daily_pv = pd.concat([daily.reset_index(drop=True), pv.reset_index(drop=True)], axis=1)
    shape_tables[entity_id] = shapes_mod.classify_daily_shape(
        daily_pv, daily_feature_tables[entity_id]
    )

shape_tables["Portfolio"][["date", "primary_shape", "is_highly_peaked", "is_unusual"]]

,date,primary_shape,is_highly_peaked,is_unusual
0,2024-07-29,multi_peak,False,False
1,2024-07-30,multi_peak,False,False
2,2024-07-31,multi_peak,False,False
3,2024-08-01,multi_peak,False,False
4,2024-08-02,multi_peak,False,False
...,...,...,...,...
722,2026-07-21,multi_peak,False,False
723,2026-07-22,multi_peak,False,False
724,2026-07-23,multi_peak,False,False
725,2026-07-24,multi_peak,False,False


## 15. Daily Profile Clustering — Absolute and Normalized (Section 19-20)

In [23]:
cluster_results = {}
analysis_entity_ids = [m["meter_id"] for m in config["meters"]] + ["Portfolio"]
for entity_id in analysis_entity_ids:
    cluster_results[(entity_id, "absolute")] = clustering_mod.cluster_daily_profiles(
        daily_profile_tables[entity_id], entity_id, value_col="demand_kw", n_clusters="auto"
    )
    cluster_results[(entity_id, "normalized")] = clustering_mod.cluster_daily_profiles(
        daily_profile_tables[entity_id], entity_id, value_col="normalized_demand", n_clusters="auto"
    )

for (entity_id, kind), r in cluster_results.items():
    if r["success"]:
        print(f"{entity_id:12s} {kind:10s} k={r['n_clusters']} silhouette={r['silhouette']}")

cluster_results[("Portfolio", "absolute")]["cluster_summary"]

KZD390965687-1 absolute   k=2 silhouette=0.5615978546055408
KZD390965687-1 normalized k=2 silhouette=0.5209183299750794
Portfolio    absolute   k=2 silhouette=0.5615978546055408
Portfolio    normalized k=2 silhouette=0.5209183299750794


,cluster_id,cluster_size,percentage_of_days,representative_peak,within_cluster_variability
0,0,269,37.3,723.287435,91.381039
1,1,452,62.7,496.818053,63.780366


## 16. Pattern Discovery (Section 21)

In [24]:
pattern_tables = {}
for entity_id in analysis_entity_ids:
    timing = patterns_mod.discover_recurring_peak_timing(daily_feature_tables[entity_id], entity_id)
    shape_pat = patterns_mod.discover_recurring_shapes(shape_tables[entity_id], entity_id)
    outliers = patterns_mod.discover_outlier_days(daily_feature_tables[entity_id], entity_id)
    pattern_tables[entity_id] = {"timing": timing, "shape": shape_pat, "outliers": outliers}

print("Portfolio outlier days:")
pattern_tables["Portfolio"]["outliers"]

Portfolio outlier days:


,pattern_id,pattern_type,description,date,metric,value,z_score,statistical_support
0,Portfolio_outlier_000,outlier_day,2024-08-29 is an outlier on daily_energy_kwh (...,2024-08-29,daily_energy_kwh,20496.28,2.629,z >= 2.5
1,Portfolio_outlier_001,outlier_day,2025-06-24 is an outlier on daily_energy_kwh (...,2025-06-24,daily_energy_kwh,20206.28,2.529,z >= 2.5
2,Portfolio_outlier_002,outlier_day,2025-06-25 is an outlier on daily_energy_kwh (...,2025-06-25,daily_energy_kwh,20877.18,2.760,z >= 2.5
3,Portfolio_outlier_003,outlier_day,2025-06-26 is an outlier on daily_energy_kwh (...,2025-06-26,daily_energy_kwh,20529.20,2.640,z >= 2.5
4,Portfolio_outlier_004,outlier_day,2025-06-30 is an outlier on daily_energy_kwh (...,2025-06-30,daily_energy_kwh,21037.74,2.816,z >= 2.5
5,Portfolio_outlier_005,outlier_day,2025-07-01 is an outlier on daily_energy_kwh (...,2025-07-01,daily_energy_kwh,21421.16,2.948,z >= 2.5
6,Portfolio_outlier_006,outlier_day,2025-07-14 is an outlier on daily_energy_kwh (...,2025-07-14,daily_energy_kwh,20306.30,2.564,z >= 2.5
7,Portfolio_outlier_007,outlier_day,2025-07-16 is an outlier on daily_energy_kwh (...,2025-07-16,daily_energy_kwh,20779.32,2.727,z >= 2.5
8,Portfolio_outlier_008,outlier_day,2025-07-17 is an outlier on daily_energy_kwh (...,2025-07-17,daily_energy_kwh,20874.54,2.759,z >= 2.5
9,Portfolio_outlier_009,outlier_day,2025-07-30 is an outlier on daily_energy_kwh (...,2025-07-30,daily_energy_kwh,20377.16,2.588,z >= 2.5


## 17. Meter Coincidence (Section 22)

In [25]:
peak_contribution = coincidence_mod.calculate_peak_contribution(
    processed_df, portfolio_meters, top_n=10
)
diversity = coincidence_mod.calculate_diversity_factor(processed_df, portfolio_meters)
interval_coincidence = coincidence_mod.calculate_interval_coincidence(processed_df, portfolio_meters)

print("Diversity factor (sum of individual peaks / aggregate peak):", diversity["diversity_factor"])
interval_coincidence

Diversity factor (sum of individual peaks / aggregate peak): 1.0


,meter_id,n_near_peak_intervals,n_coincident_with_any_other,coincidence_rate
0,KZD390965687-1,86,0,0.0


## 18. Stage 2 Summary and Export (Section 51-52 — Stage 2 subset)

In [26]:
print("=" * 72)
print("STAGE 2 ANALYTICAL SUMMARY")
print("=" * 72)
print(f"Change-point (weekday cooling, 3P) models fit: {list(change_point_results.keys())}")
print(f"Change-point (weekday heating+cooling, 5P) models fit: {list(change_point_5p_results.keys())}")
print("Change-point model family selection (2P/3P heating/3P cooling/4P/5P) per meter:")
for meter_id, r in change_point_family_results.items():
    print(f"  {meter_id}: {r['selected_model']}")
print(f"Peak events built (threshold={demand_thresholds[0]} kW): "
      f"{[(k, len(v)) for k, v in peak_events_by_entity.items()]}")
print("Primary shape distribution (Portfolio):")
print(shape_tables["Portfolio"]["primary_shape"].value_counts().to_string())
print(f"Diversity factor (portfolio): {diversity['diversity_factor']:.3f}")
n_patterns = sum(
    len(t["timing"]) + len(t["shape"]) + len(t["outliers"]) for t in pattern_tables.values()
)
print(f"Total discovered patterns across meters/portfolio: {n_patterns}")
print("=" * 72)
print("Stage 2 complete. Ready for Stage 3 (opportunity/scenario engine, TOU, searches, exports, viz).")

STAGE 2 ANALYTICAL SUMMARY
Change-point (weekday cooling, 3P) models fit: ['KZD390965687-1']
Change-point (weekday heating+cooling, 5P) models fit: ['KZD390965687-1']
Change-point model family selection (2P/3P heating/3P cooling/4P/5P) per meter:
  KZD390965687-1: 4p
Peak events built (threshold=500 kW): [('KZD390965687-1', 493), ('Portfolio', 493)]
Primary shape distribution (Portfolio):
primary_shape
multi_peak           722
insufficient_data      5
Diversity factor (portfolio): 1.000
Total discovered patterns across meters/portfolio: 110
Stage 2 complete. Ready for Stage 3 (opportunity/scenario engine, TOU, searches, exports, viz).


In [27]:
for entity_id, events in peak_events_by_entity.items():
    events.to_csv(output_dir / "peaks" / f"peak_events_{entity_id}.csv", index=False)
for (entity_id, kind), r in cluster_results.items():
    if r["success"]:
        r["cluster_summary"].to_csv(output_dir / "clusters" / f"clusters_{entity_id}_{kind}.csv", index=False)
for entity_id, tset in pattern_tables.items():
    for pat_type, df_pat in tset.items():
        if len(df_pat):
            df_pat.to_csv(output_dir / "patterns" / f"patterns_{entity_id}_{pat_type}.csv", index=False)

print(f"Stage 2 outputs exported to: {output_dir}")

Stage 2 outputs exported to: /Users/egeriicw/Personal-Projects/load_flexibility_and_der_analysis/output
